# Photo Pipeline v2 — 3-Phase Analysis

**Phase 1 (Gemini)** → factual extraction  
**Phase 2 (Claude)** → artistic analysis  
**Phase 3 (Claude)** → SEO + location metadata

In [ ]:
# Cell: Setup & Schema

# ── imports ──────────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(override=True)

import duckdb, anthropic
import google.genai as genai
from google.genai import types as genai_types
from google.genai import errors as genai_errors
import base64, hashlib, time, json as _json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

# ── constants ─────────────────────────────────────────────────────────────────
DB_PATH  = "./outputs/photos.duckdb"
RAW_DIR  = Path("./data/raw")

# Set to None to process all photos, or a list of filenames:
#   PROCESS_PHOTOS = ["Kinan.Sweidan-21.JPG"]
PROCESS_PHOTOS = None

# ── DB + schema ───────────────────────────────────────────────────────────────
con = duckdb.connect(DB_PATH)

con.execute("""
    CREATE TABLE IF NOT EXISTS phase_runs (
        run_id     INTEGER PRIMARY KEY,
        notes      VARCHAR,
        p1_prompt  VARCHAR,
        p2_prompt  VARCHAR,
        p3_prompt  VARCHAR,
        created_at TIMESTAMP DEFAULT current_timestamp
    )
""")

con.execute("""
    CREATE TABLE IF NOT EXISTS phase_outputs (
        photo_id              VARCHAR,
        run_id                INTEGER NOT NULL DEFAULT 1,

        -- Phase 1 (Gemini) — factual extraction
        p1_raw                VARCHAR,
        subject_count         INTEGER,
        subject_position      VARCHAR,
        viewpoint             VARCHAR,
        depth_layers          INTEGER,
        horizon_present       BOOLEAN,
        leading_lines         BOOLEAN,
        leading_line_dir      VARCHAR,
        frame_within_frame    BOOLEAN,
        symmetry              VARCHAR,
        subject_space_ratio   VARCHAR,
        motion                VARCHAR,
        tonal_key             VARCHAR,
        contrast              VARCHAR,
        shadow_as_subject     BOOLEAN,
        reflection_present    BOOLEAN,
        light_direction       VARCHAR,
        highlight_rendering   VARCHAR,
        grain_texture         VARCHAR,
        objects               VARCHAR,
        p1_latency_ms         INTEGER,

        -- Phase 2 (Claude) — artistic analysis
        p2_raw                VARCHAR,
        caption               VARCHAR,
        mood                  VARCHAR,
        light_quality         VARCHAR,
        compositional_tension VARCHAR,
        photographic_tradition VARCHAR,
        background            VARCHAR,
        style_fingerprint     VARCHAR,
        p2_latency_ms         INTEGER,

        -- Phase 3 (Claude) — SEO + location
        p3_raw                VARCHAR,
        seo_filename          VARCHAR,
        alt_text              VARCHAR,
        title_tag             VARCHAR,
        tags                  VARCHAR,
        content_location      VARCHAR,
        p3_latency_ms         INTEGER,

        created_at            TIMESTAMP DEFAULT current_timestamp,

        PRIMARY KEY (photo_id, run_id)
    )
""")

# ── load photos ───────────────────────────────────────────────────────────────
def file_hash(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()[:16]

hash_to_path = {
    file_hash(str(p)): p
    for p in sorted(p for ext in ("*.jpg", "*.JPG", "*.jpeg", "*.JPEG")
                    for p in RAW_DIR.glob(ext))
}

if PROCESS_PHOTOS is None:
    target_photos = hash_to_path
else:
    target_names = set(PROCESS_PHOTOS)
    target_photos = {k: v for k, v in hash_to_path.items() if v.name in target_names}
    missing = target_names - {v.name for v in target_photos.values()}
    if missing:
        print(f"  [warn] not found in RAW_DIR: {missing}")

CURRENT_RUN_ID = None  # set by running the New Run cell

# ── summary ───────────────────────────────────────────────────────────────────
runs = con.execute("SELECT run_id, notes FROM phase_runs ORDER BY run_id").fetchall()
counts = con.execute(
    "SELECT run_id, COUNT(*) FROM phase_outputs GROUP BY run_id ORDER BY run_id"
).fetchall()

print(f"Photos found: {len(hash_to_path)}  |  Targeted: {len(target_photos)}")
if runs:
    print(f"phase_runs: {[f'#{r} {n}' for r, n in runs]}")
print(f"phase_outputs rows: {counts or 'empty'}")


In [ ]:
# Cell: Prompts

P1_PROMPT = """You are a computer vision system performing structured analysis of a black and white street photograph.
Extract ONLY what is directly observable. No interpretation, no subjective language.
If a value is ambiguous, use the closest discrete option from the allowed values.

Return each field on its own line in FIELD: value format. No preamble. No explanation.

subject_count: <integer>
subject_position: left|center|right|edge
viewpoint: eye-level|low|high|overhead
depth_layers: <integer 1-3>
horizon_present: true|false
leading_lines: true|false
leading_line_direction: diagonal|converging|horizontal|vertical|none
frame_within_frame: true|false
symmetry: symmetrical|asymmetrical
subject_to_space_ratio: subject-dominant|balanced|space-dominant
motion: frozen|blurred|implied|none
tonal_key: high|mid|low
contrast: flat|medium|high
shadow_as_subject: true|false
reflection_present: true|false
light_direction: front|side|back|overhead|diffuse
highlight_rendering: blown|retained|glowing
grain_texture: none|light|medium|heavy
objects: comma-separated list of visible objects, no interpretation"""

P2_PROMPT = """You are an expert in 20th-century street photography with deep knowledge of Cartier-Bresson,
Vivian Maier, Daido Moriyama, and the black-and-white humanist tradition.

You will receive a black and white street photograph and a factual analysis.
Use the factual analysis as grounding. Build interpretation on top of those facts --
do not contradict them and do not repeat them verbatim.

Think step by step before writing:
- What is the decisive moment or tension in this image?
- How does the tonal rendering serve the mood?
- What is the relationship between figure and environment?
- What photographic tradition or aesthetic does this most resemble?

Return each field on its own line in FIELD: value format. No preamble. No explanation.

caption: one sentence, present tense, no photographer named, evocative but grounded
mood: exactly 3 adjectives, comma-separated, no repetition with caption
light_quality: one phrase describing how light functions in this image
compositional_tension: one sentence on what creates visual energy
photographic_tradition: closest named tradition or photographer aesthetic
background: 2-3 sentences describing environment, atmosphere, and context
style_fingerprint: comma-separated list of exactly 5 stylistic descriptors for similarity search

FACTUAL ANALYSIS:
{p1_output}"""

P3_PROMPT = """You are an SEO specialist for a fine art black and white street photography portfolio.
Your audience is: art collectors, editorial photo editors, stock photo buyers, and photography enthusiasts.

Rules:
- Tags must be unique -- no synonym pairs (e.g., not both "urban" and "city")
- Tags must span multiple search intents: subject, style, mood, use-case, era-feel
- alt_text must be under 125 characters
- seo_filename must be kebab-case, 4-6 words, no generic terms like "photo" or "image"
- List tags ordered by search volume potential, high to low

Location context: These photographs were taken in Chicago, Illinois.
Use Chicago, IL as the default for content_location. Only specify a
neighborhood if confirmed by a visible sign or landmark.

Return each field on its own line in FIELD: value format. No preamble. No explanation.

seo_filename: kebab-case-4-to-6-words
alt_text: under 125 chars, descriptive, keyword-rich, no "photo of" prefix
title_tag: under 60 chars, gallery-appropriate
tags: exactly 15 unique tags, comma-separated, ordered high to low search volume
content_location: Chicago, IL or specific neighborhood

FACTUAL ANALYSIS:
{p1_output}

ARTISTIC ANALYSIS:
{p2_output}"""


In [ ]:
# Cell: Parsers & Savers

def parse_kv(output: str) -> dict:
    result = {}
    for line in output.strip().splitlines():
        if ": " in line:
            key, _, value = line.partition(": ")
            result[key.strip()] = value.strip()
    return result

def parse_bool(v):
    return v.lower() == "true" if v else None

def parse_int(v):
    try:
        return int(v)
    except (TypeError, ValueError):
        return None


def save_phase1(photo_id: str, run_id: int, raw: str, parsed: dict, latency_ms: int):
    # Ensure row exists
    con.execute("""
        INSERT INTO phase_outputs (photo_id, run_id)
        VALUES (?, ?)
        ON CONFLICT (photo_id, run_id) DO NOTHING
    """, [photo_id, run_id])
    con.execute("""
        UPDATE phase_outputs SET
            p1_raw             = ?,
            subject_count      = ?,
            subject_position   = ?,
            viewpoint          = ?,
            depth_layers       = ?,
            horizon_present    = ?,
            leading_lines      = ?,
            leading_line_dir   = ?,
            frame_within_frame = ?,
            symmetry           = ?,
            subject_space_ratio= ?,
            motion             = ?,
            tonal_key          = ?,
            contrast           = ?,
            shadow_as_subject  = ?,
            reflection_present = ?,
            light_direction    = ?,
            highlight_rendering= ?,
            grain_texture      = ?,
            objects            = ?,
            p1_latency_ms      = ?
        WHERE photo_id = ? AND run_id = ?
    """, [
        raw,
        parse_int(parsed.get("subject_count")),
        parsed.get("subject_position"),
        parsed.get("viewpoint"),
        parse_int(parsed.get("depth_layers")),
        parse_bool(parsed.get("horizon_present")),
        parse_bool(parsed.get("leading_lines")),
        parsed.get("leading_line_direction"),
        parse_bool(parsed.get("frame_within_frame")),
        parsed.get("symmetry"),
        parsed.get("subject_to_space_ratio"),
        parsed.get("motion"),
        parsed.get("tonal_key"),
        parsed.get("contrast"),
        parse_bool(parsed.get("shadow_as_subject")),
        parse_bool(parsed.get("reflection_present")),
        parsed.get("light_direction"),
        parsed.get("highlight_rendering"),
        parsed.get("grain_texture"),
        parsed.get("objects"),
        latency_ms,
        photo_id, run_id,
    ])


def save_phase2(photo_id: str, run_id: int, raw: str, parsed: dict, latency_ms: int):
    con.execute("""
        UPDATE phase_outputs SET
            p2_raw                 = ?,
            caption                = ?,
            mood                   = ?,
            light_quality          = ?,
            compositional_tension  = ?,
            photographic_tradition = ?,
            background             = ?,
            style_fingerprint      = ?,
            p2_latency_ms          = ?
        WHERE photo_id = ? AND run_id = ?
    """, [
        raw,
        parsed.get("caption"),
        parsed.get("mood"),
        parsed.get("light_quality"),
        parsed.get("compositional_tension"),
        parsed.get("photographic_tradition"),
        parsed.get("background"),
        parsed.get("style_fingerprint"),
        latency_ms,
        photo_id, run_id,
    ])


def save_phase3(photo_id: str, run_id: int, raw: str, parsed: dict, latency_ms: int):
    con.execute("""
        UPDATE phase_outputs SET
            p3_raw           = ?,
            seo_filename     = ?,
            alt_text         = ?,
            title_tag        = ?,
            tags             = ?,
            content_location = ?,
            p3_latency_ms    = ?
        WHERE photo_id = ? AND run_id = ?
    """, [
        raw,
        parsed.get("seo_filename"),
        parsed.get("alt_text"),
        parsed.get("title_tag"),
        parsed.get("tags"),
        parsed.get("content_location"),
        latency_ms,
        photo_id, run_id,
    ])

print("Parsers and savers ready.")


In [ ]:
# Cell: New Run
# Edit RUN_NOTES to describe what changed, then run this cell before the phase cells.

RUN_NOTES = "v1 - initial 3-phase run"

next_id = (con.execute("SELECT COALESCE(MAX(run_id), 0) + 1 FROM phase_runs").fetchone()[0])
con.execute(
    "INSERT INTO phase_runs (run_id, notes, p1_prompt, p2_prompt, p3_prompt) VALUES (?, ?, ?, ?, ?)",
    [next_id, RUN_NOTES, P1_PROMPT, P2_PROMPT, P3_PROMPT],
)
CURRENT_RUN_ID = next_id
print(f"CURRENT_RUN_ID = {CURRENT_RUN_ID}  |  notes: {RUN_NOTES}")


In [ ]:
# Cell: Run Phase 1 — Gemini (Factual Extraction)

def run_phase1(image_path: str) -> tuple[str, int]:
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    client = genai.Client()
    for attempt in range(4):
        try:
            t0 = time.time()
            response = client.models.generate_content(
                model="gemini-2.5-pro",
                contents=[
                    genai_types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
                    P1_PROMPT,
                ],
            )
            latency_ms = int((time.time() - t0) * 1000)
            return response.text.strip(), latency_ms
        except genai_errors.ServerError:
            if attempt == 3:
                raise
            wait = 2 ** attempt * 5
            print(f"  [retry {attempt+1}] 503 -- waiting {wait}s ...")
            time.sleep(wait)


assert CURRENT_RUN_ID is not None, "Run the 'New Run' cell first to set CURRENT_RUN_ID"

done = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM phase_outputs WHERE p1_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}

for photo_id, path in target_photos.items():
    if photo_id in done:
        print(f"  skip  {path.name}")
        continue
    raw, ms = run_phase1(str(path))
    parsed = parse_kv(raw)
    save_phase1(photo_id, CURRENT_RUN_ID, raw, parsed, ms)
    print(f"  done  {path.name} | tonal={parsed.get('tonal_key')} contrast={parsed.get('contrast')} | {ms}ms")

print("\nPhase 1 complete.")


In [ ]:
# Cell: Run Phase 2 — Claude (Artistic Analysis)
# Requires Phase 1 to be complete for each photo.

def run_phase2(image_path: str, p1_output: str) -> tuple[str, int]:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    client = anthropic.Anthropic()
    prompt = P2_PROMPT.format(p1_output=p1_output)
    t0 = time.time()
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}},
                {"type": "text", "text": prompt},
            ],
        }],
    )
    latency_ms = int((time.time() - t0) * 1000)
    return response.content[0].text.strip(), latency_ms


assert CURRENT_RUN_ID is not None, "Run the 'New Run' cell first to set CURRENT_RUN_ID"

# Check Phase 1 coverage
p1_complete = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM phase_outputs WHERE p1_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}
missing_p1 = [pid for pid in target_photos if pid not in p1_complete]
assert not missing_p1, f"Phase 1 incomplete for {len(missing_p1)} photo(s). Run Phase 1 first."

done = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM phase_outputs WHERE p2_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}

for photo_id, path in target_photos.items():
    if photo_id in done:
        print(f"  skip  {path.name}")
        continue
    p1_raw = con.execute(
        "SELECT p1_raw FROM phase_outputs WHERE photo_id = ? AND run_id = ?",
        [photo_id, CURRENT_RUN_ID],
    ).fetchone()[0]
    raw, ms = run_phase2(str(path), p1_raw)
    parsed = parse_kv(raw)
    save_phase2(photo_id, CURRENT_RUN_ID, raw, parsed, ms)
    print(f"  done  {path.name} | {parsed.get('caption','')[:70]} | {ms}ms")

print("\nPhase 2 complete.")


In [ ]:
# Cell: Run Phase 3 — Claude (SEO + Location)
# Requires Phase 2 to be complete for each photo.

def run_phase3(image_path: str, p1_output: str, p2_output: str) -> tuple[str, int]:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    client = anthropic.Anthropic()
    prompt = P3_PROMPT.format(p1_output=p1_output, p2_output=p2_output)
    t0 = time.time()
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}},
                {"type": "text", "text": prompt},
            ],
        }],
    )
    latency_ms = int((time.time() - t0) * 1000)
    return response.content[0].text.strip(), latency_ms


assert CURRENT_RUN_ID is not None, "Run the 'New Run' cell first to set CURRENT_RUN_ID"

# Check Phase 2 coverage
p2_complete = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM phase_outputs WHERE p2_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}
missing_p2 = [pid for pid in target_photos if pid not in p2_complete]
assert not missing_p2, f"Phase 2 incomplete for {len(missing_p2)} photo(s). Run Phase 2 first."

done = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM phase_outputs WHERE p3_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}

for photo_id, path in target_photos.items():
    if photo_id in done:
        print(f"  skip  {path.name}")
        continue
    row = con.execute(
        "SELECT p1_raw, p2_raw FROM phase_outputs WHERE photo_id = ? AND run_id = ?",
        [photo_id, CURRENT_RUN_ID],
    ).fetchone()
    raw, ms = run_phase3(str(path), row[0], row[1])
    parsed = parse_kv(raw)
    save_phase3(photo_id, CURRENT_RUN_ID, raw, parsed, ms)
    print(f"  done  {path.name} | {parsed.get('seo_filename','')[:50]} | loc={parsed.get('content_location')} | {ms}ms")

print("\nPhase 3 complete.")


In [ ]:
# Cell: Visual Review
# Shows each photo with its 3-phase results side-by-side.

def show_review(photo_id: str, path):
    row = con.execute("""
        SELECT caption, mood, light_quality, compositional_tension,
               photographic_tradition, tonal_key, contrast, grain_texture,
               seo_filename, alt_text, title_tag, tags, content_location,
               p1_latency_ms, p2_latency_ms, p3_latency_ms
        FROM phase_outputs
        WHERE photo_id = ? AND run_id = ?
    """, [photo_id, CURRENT_RUN_ID]).fetchone()

    if row is None:
        print(f"No data for {path.name} in run {CURRENT_RUN_ID}")
        return

    (caption, mood, light_quality, comp_tension, tradition,
     tonal_key, contrast, grain, seo_fn, alt_text, title_tag, tags,
     location, ms1, ms2, ms3) = row

    fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                             gridspec_kw={"width_ratios": [1, 1.6]})
    img = Image.open(path)
    axes[0].imshow(img, cmap="gray" if img.mode == "L" else None)
    axes[0].axis("off")
    axes[0].set_title(path.name, fontsize=9, color="#666")

    info = []
    info.append(f"── Phase 1 · Gemini ({ms1}ms) ──")
    info.append(f"tonal: {tonal_key}  contrast: {contrast}  grain: {grain}")
    info.append(f"tradition: {tradition}")
    info.append("")
    info.append(f"── Phase 2 · Claude ({ms2}ms) ──")
    if caption:
        info.append(f"caption: {caption}")
    if mood:
        info.append(f"mood: {mood}")
    if light_quality:
        info.append(f"light: {light_quality}")
    if comp_tension:
        info.append(f"tension: {comp_tension}")
    info.append("")
    info.append(f"── Phase 3 · Claude ({ms3}ms) ──")
    if title_tag:
        info.append(f"title: {title_tag}")
    if seo_fn:
        info.append(f"slug: {seo_fn}")
    if location:
        info.append(f"location: {location}")
    if alt_text:
        info.append(f"alt: {alt_text}")
    if tags:
        tag_list = [t.strip() for t in tags.split(",")]
        info.append(f"tags: {', '.join(tag_list[:8])}{'...' if len(tag_list) > 8 else ''}")

    axes[1].axis("off")
    axes[1].text(0.02, 0.98, "\n".join(info),
                 transform=axes[1].transAxes,
                 va="top", ha="left", fontsize=9,
                 fontfamily="monospace",
                 wrap=True)

    plt.tight_layout()
    plt.show()


assert CURRENT_RUN_ID is not None, "Run the 'New Run' cell first to set CURRENT_RUN_ID"

have = {
    row[0] for row in
    con.execute(
        "SELECT photo_id FROM phase_outputs WHERE p3_raw IS NOT NULL AND run_id = ?",
        [CURRENT_RUN_ID],
    ).fetchall()
}

reviewed = [pid for pid in hash_to_path if pid in have]
print(f"Showing {len(reviewed)} completed photos:")
for pid in reviewed:
    show_review(pid, hash_to_path[pid])


In [ ]:
# Cell: Analytics

assert CURRENT_RUN_ID is not None, "Run the 'New Run' cell first to set CURRENT_RUN_ID"

rows = con.execute("""
    SELECT photo_id, tonal_key, contrast, grain_texture,
           photographic_tradition, mood, content_location,
           p1_latency_ms, p2_latency_ms, p3_latency_ms
    FROM phase_outputs
    WHERE run_id = ?
    ORDER BY photo_id
""", [CURRENT_RUN_ID]).fetchall()

if not rows:
    print("No data for this run yet. Run all 3 phases first.")
else:
    total = len(rows)
    print(f"Run #{CURRENT_RUN_ID}  —  {total} photo(s) fully processed\n")

    # ── Field completeness ────────────────────────────────────────────────────
    fields_to_check = ["tonal_key", "contrast", "grain_texture",
                       "photographic_tradition", "mood", "content_location"]
    col_idx = {
        "tonal_key": 1, "contrast": 2, "grain_texture": 3,
        "photographic_tradition": 4, "mood": 5, "content_location": 6,
    }
    print("Field completeness:")
    for field in fields_to_check:
        filled = sum(1 for r in rows if r[col_idx[field]])
        print(f"  {field:<26} {filled}/{total}  ({100*filled//total}%)")

    # ── Tonal key distribution ────────────────────────────────────────────────
    from collections import Counter
    tonal_counts = Counter(r[1] for r in rows if r[1])
    print(f"\nTonal key: {dict(tonal_counts)}")

    # ── Photographic tradition breakdown ─────────────────────────────────────
    tradition_counts = Counter(r[4] for r in rows if r[4])
    print("\nPhotographic traditions:")
    for trad, cnt in tradition_counts.most_common():
        print(f"  {cnt:2d}x  {trad}")

    # ── Latency summary ───────────────────────────────────────────────────────
    ms1 = [r[7] for r in rows if r[7]]
    ms2 = [r[8] for r in rows if r[8]]
    ms3 = [r[9] for r in rows if r[9]]
    if ms1:
        print(f"\nLatency avg  P1={sum(ms1)//len(ms1)}ms  P2={sum(ms2)//len(ms2) if ms2 else '—'}ms  P3={sum(ms3)//len(ms3) if ms3 else '—'}ms")

    # ── Tonal key bar chart ───────────────────────────────────────────────────
    if tonal_counts:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        keys = ["high", "mid", "low"]
        vals = [tonal_counts.get(k, 0) for k in keys]
        axes[0].bar(keys, vals, color=["#e5e7eb", "#6b7280", "#1f2937"])
        axes[0].set_title("Tonal Key Distribution")
        axes[0].set_ylabel("Photos")

        if tradition_counts:
            trads = list(tradition_counts.keys())[:8]
            tcnts = [tradition_counts[t] for t in trads]
            axes[1].barh(trads, tcnts, color="#3b82f6")
            axes[1].set_title("Photographic Tradition")
            axes[1].set_xlabel("Photos")

        plt.tight_layout()
        plt.show()
